# Tiny ImageNet research figures

Purpose: render actual training, fixed-subset diagnostics, frozen-probe comparisons and separately labeled runtime/memory workloads. Inputs are completed run metric rows; missing measurements stay visibly missing. Outputs are Matplotlib figures and PNG/PDF exports through `save_figure`. Expected runtime: seconds. No training or data download occurs on import. Resume: regenerate figures from the saved metric tables; no model checkpoint is modified.


In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
from pathlib import Path
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle

COLORS={'vit':'#b96a23','fmm':'#245b80'}

def setup_style():
    plt.rcParams.update({'figure.figsize':(10,4),'figure.dpi':110,'font.size':10,
                         'axes.spines.top':False,'axes.spines.right':False,'axes.grid':False})

def save_figure(fig,artifact,name):
    directory=Path(artifact)/'figures';directory.mkdir(exist_ok=True,parents=True)
    fig.savefig(directory/(name+'.png'),dpi=150,bbox_inches='tight')
    fig.savefig(directory/(name+'.pdf'),bbox_inches='tight')
    plt.show()

def pretrain_probe_diagram():
    fig,ax=plt.subplots(figsize=(11,5));ax.set(xlim=(0,12),ylim=(0,5));ax.axis('off')
    boxes=[(.2,3.3,2.0,.85,'Training images\npaired augmentation'),(2.8,3.3,2.3,.85,'Visible student\n+ full-image teacher'),
           (5.7,3.3,2.6,.85,'Latent prediction\nSmoothL1 + EMA'),(9,3.3,2.4,.85,'Saved encoder\ninitial and final'),
           (.2,.8,2,.85,'Train/dev images\ndeterministic transform'),(2.8,.8,2.3,.85,'Frozen EMA encoder\nmean patch features'),
           (5.7,.8,2.6,.85,'Linear classifier only\ntrain-pool labels'),(9,.8,2.4,.85,'Development top-1/5\ntrained vs initialized')]
    for x,y,w,h,label in boxes:
        ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=.05',facecolor='#eef3f7',edgecolor='#245b80'))
        ax.text(x+w/2,y+h/2,label,ha='center',va='center',fontsize=9.5)
    for y in (3.725,1.225):
        for x0,x1 in ((2.2,2.8),(5.1,5.7),(8.3,9)):
            ax.annotate('',xy=(x1,y),xytext=(x0,y),arrowprops={'arrowstyle':'->','color':'#333333'})
    ax.text(.2,4.55,'Self-supervised pretraining — labels excluded',weight='bold')
    ax.text(.2,2.08,'Frozen-feature evaluation — encoder never optimized',weight='bold')
    ax.text(.2,.15,'Held-out source validation stays closed in smoke mode. The predictor is absent from feature evaluation.',fontsize=10)
    fig.tight_layout();return fig

def examples_figure(batch,config,count=3):
    images=batch['images'].detach().cpu()
    mean=torch.tensor(config['preprocessing']['mean'])[:,None,None]
    std=torch.tensor(config['preprocessing']['std'])[:,None,None]
    images=(images*std+mean).clamp(0,1)
    masks=batch.get('masks',batch)
    count=min(count,len(images));patch=config['model']['patch_size'];rows=images.shape[-2]//patch;cols=images.shape[-1]//patch
    fig,axes=plt.subplots(count,3,figsize=(9,2.7*count),squeeze=False)
    for index in range(count):
        image=images[index].permute(1,2,0).numpy()
        context=masks['context_indices'][index][masks['context_valid'][index]].cpu().numpy()
        visible=np.zeros(rows*cols,dtype=bool);visible[context]=True
        pixels=np.repeat(np.repeat(visible.reshape(rows,cols),patch,axis=0),patch,axis=1)
        axes[index,0].imshow(image);axes[index,1].imshow(image*pixels[:,:,None]);axes[index,2].imshow(image)
        union=set()
        for ids,valid in zip(masks['target_indices'],masks['target_valid']):union.update(ids[index][valid[index]].cpu().tolist())
        for idx in union:
            axes[index,2].add_patch(Rectangle(((idx%cols)*patch-.5,(idx//cols)*patch-.5),patch,patch,fill=False,edgecolor='#ffdf70',linewidth=.8,hatch='//'))
        axes[index,0].set_title('Actual augmented input')
        axes[index,1].set_title(f'Visible context: {len(context)}/{rows*cols} patches')
        axes[index,2].set_title(f'Target union: {len(union)} patches')
        for ax in axes[index]:ax.set(xticks=[],yticks=[])
    fig.suptitle('Real Tiny ImageNet inputs; identical image and masks for FMM and ViT')
    fig.tight_layout();return fig

def _kind(row):
    return str(row.get('model_id',row.get('encoder_kind',row.get('kind','')))).split('_')[0]

def _step(row): return row.get('successful_updates',row.get('step',row.get('update')))

def _finish_axis(ax,xlabel,ylabel):
    ax.set(xlabel=xlabel,ylabel=ylabel);ax.grid(axis='y',alpha=.2)
    handles,labels=ax.get_legend_handles_labels()
    if handles: ax.legend(fontsize=9,frameon=False)
    else: ax.text(.5,.5,'NOT RUN / not recorded',transform=ax.transAxes,ha='center',va='center',color='#666666')

def training_figure(rows,run_mode='smoke'):
    fig,axes=plt.subplots(2,2,figsize=(11,7.2))
    for kind in ('vit','fmm'):
        chosen=sorted([r for r in rows if _kind(r)==kind and _step(r) is not None],key=_step)
        for ax,key,label in ((axes[0,0],'loss','JEPA loss'),(axes[0,1],'gradient_norm_preclip','Pre-clipping gradient norm')):
            valid=[r for r in chosen if r.get(key) is not None]
            if valid:ax.plot([_step(r) for r in valid],[r[key] for r in valid],label=kind.upper(),marker='o' if kind=='fmm' else 's',markersize=3,color=COLORS[kind],linestyle='-' if kind=='fmm' else '--')
    chosen=sorted([r for r in rows if _kind(r)=='fmm' and _step(r) is not None],key=_step)
    for prefix,style,marker in (('near','--','s'),('far','-','o')):
        for ax,key in ((axes[1,0],prefix+'_norm'),(axes[1,1],prefix+'_gradient_norm_postclip')):
            valid=[r for r in chosen if r.get(key) is not None]
            if valid:ax.plot([_step(r) for r in valid],[r[key] for r in valid],label='FMM '+prefix,marker=marker,markersize=3,color=COLORS['fmm'],linestyle=style,markerfacecolor='white' if prefix=='near' else COLORS['fmm'])
    labels=['JEPA loss','Pre-clipping gradient norm','FMM message norm','FMM branch gradient norm after clipping']
    for ax,label in zip(axes.flat,labels):_finish_axis(ax,'Successful optimizer updates',label)
    fig.suptitle(('Integration-only smoke' if run_mode=='smoke' else 'Real-data pilot')+': training and FMM branch activity')
    fig.text(.5,.01,'Masks and EMA targets change during training. Nonzero branch activity does not establish useful representations.',ha='center',fontsize=10)
    fig.tight_layout(rect=(0,.04,1,.96));return fig

def diagnostics_figure(rows,run_mode='smoke'):
    # The runner stores callback results under results; workflow CSV rows may be flat.
    flattened=[{**r,**r.get('results',{})} for r in rows]
    fig,axes=plt.subplots(2,2,figsize=(11,7.2))
    specifications=[('feature_std_mean','Across-image feature standard deviation'),('effective_rank','Covariance effective rank'),
                    ('between_image_cosine_mean','Between-image cosine similarity'),
                    ('matching_patch_across_images_std_mean','Matching-patch variation across images')]
    for kind in ('vit','fmm'):
        selected=sorted([r for r in flattened if _kind(r)==kind and _step(r) is not None],key=_step)
        for ax,(key,label) in zip(axes.flat,specifications):
            valid=[r for r in selected if r.get(key) is not None]
            if valid:ax.plot([_step(r) for r in valid],[r[key] for r in valid],marker='o' if kind=='fmm' else 's',linestyle='-' if kind=='fmm' else '--',color=COLORS[kind],label=kind.upper())
    for ax,(_,label) in zip(axes.flat,specifications):_finish_axis(ax,'Successful optimizer updates',label)
    counts=sorted({r['diagnostic_examples'] for r in flattened if r.get('diagnostic_examples') is not None})
    ceilings=sorted({r['effective_rank_ceiling'] for r in flattened if r.get('effective_rank_ceiling') is not None})
    count_label=', '.join(map(str,counts)) if counts else 'not recorded'
    ceiling_label=', '.join(map(str,ceilings)) if ceilings else 'not recorded'
    fig.suptitle(f'Fixed development subset: n = {count_label}; raw embeddings before standardization')
    caveat='Smoke learning remains inconclusive.' if run_mode=='smoke' else 'Interpret with trained-versus-initial probe results.'
    fig.text(.5,.01,f'Effective-rank ceiling: {ceiling_label}. Dispersion measures are diagnostic. {caveat}',ha='center',fontsize=10)
    fig.tight_layout(rect=(0,.04,1,.96));return fig

def _control_summary(value):
    summary=value.get('summary',value)
    return summary,summary.get('metrics',summary)

def probe_figure(controls,run_mode='smoke'):
    fig,axes=plt.subplots(1,2,figsize=(11,4.5))
    names=['vit_untrained','vit_trained','fmm_untrained','fmm_trained']
    labels=['ViT\nuntrained','ViT\ntrained','FMM\nuntrained','FMM\ntrained']
    normalized={name:_control_summary(value) for name,value in controls.items()}
    # Backward compatibility with the root's early plotting stub; scientific meaning is identical.
    for kind in ('vit','fmm'):
        if kind+'_initial' in normalized and kind+'_untrained' not in normalized:normalized[kind+'_untrained']=normalized[kind+'_initial']
    counts=[]
    for ax,metric in zip(axes,('top1','top5')):
        values=[]
        for i,name in enumerate(names):
            summary,metrics=normalized.get(name,({},{}))
            if metric not in metrics or metrics[metric] is None:
                ax.text(i,.03,'NOT RUN',transform=ax.get_xaxis_transform(),rotation=90,ha='center',va='bottom',color='#666666');continue
            value=float(metrics[metric])
            if not np.isfinite(value) or not 0<=value<=1:raise ValueError('Probe accuracies must be finite fractions in [0,1]')
            color=COLORS[name.split('_')[0]];initial=name.endswith('untrained')
            bar=ax.bar(i,100*value,color='white' if initial else color,edgecolor=color,hatch='///' if initial else None,linewidth=1.5)
            ax.bar_label(bar,fmt='%.2f%%',padding=3,fontsize=10);values.append(100*value)
            if metrics.get('examples') is not None:counts.append(int(metrics['examples']))
        ax.set(xticks=range(4),xticklabels=labels,ylabel=f'Development {metric.replace("top","top-")} accuracy (%)',ylim=(0,max(1,max(values,default=0)*1.32)))
        ax.grid(axis='y',alpha=.2)
    fig.suptitle('Frozen linear probes: each trained encoder compared with its own initialization')
    count=', '.join(map(str,sorted(set(counts)))) if counts else 'not recorded'
    caveat='Integration-only smoke; learning is inconclusive.' if run_mode=='smoke' else 'One paired seed; development selects probe epochs.'
    fig.text(.5,.02,f'Identical probe budgets; development n = {count}. {caveat}',ha='center',fontsize=10)
    fig.tight_layout(rect=(0,.07,1,.96));return fig

def _phase(row):
    phase=str(row.get('phase',row.get('scope','')))
    if phase=='pretraining' or 'complete_jepa_step_seconds' in row:return 'pretraining'
    if 'feature' in phase:return 'feature_extraction'
    if 'probe' in phase:return 'probe_fitting'
    if 'encoder' in phase:return 'encoder_forward'
    return phase

def _performance_label(row):
    model=str(row.get('model_id',row.get('encoder_kind',row.get('kind','unlabeled'))))
    parts=model.replace('_initial','_untrained').split('_');label={'vit':'ViT','fmm':'FMM'}.get(parts[0],parts[0])
    if len(parts)>1:label+=' '+ ' '.join(parts[1:])
    split=row.get('split')
    if split:label+=' · '+str(split).replace('development','dev')
    return label

def _timing_value(row,phase):
    if row.get('status') in ('NOT_RUN','FAIL'):return None,None,None
    if phase=='pretraining':
        stats=row.get('complete_jepa_step_seconds')
        if not stats:return None,None,None
        median=float(stats['median']);return median,max(0,median-float(stats['p25'])),max(0,float(stats['p75'])-median)
    if phase=='encoder_forward':
        median=row.get('median_ms')
        if median is None:median=1000*row['median_seconds'] if row.get('median_seconds') is not None else None
        if median is None:median=1000*row['median_batch_seconds'] if row.get('median_batch_seconds') is not None else None
        mad=row.get('mad_ms')
        if mad is None:mad=1000*row['mad_seconds'] if row.get('mad_seconds') is not None else None
        return median,mad,mad
    return row.get('wall_seconds'),None,None

def performance_figure(rows,run_mode='smoke'):
    """Separate stage workloads. Bars start at zero; unavailable timings/memory stay missing."""
    phase_order=['pretraining','encoder_forward','feature_extraction','probe_fitting']
    grouped={phase:[r for r in rows if _phase(r)==phase] for phase in phase_order}
    active=[phase for phase in phase_order if grouped[phase]]
    if not active:active=['pretraining']
    heights=[max(2.5,.38*len(grouped[phase])+1.25) for phase in active]
    fig,axes=plt.subplots(len(active),2,figsize=(13,sum(heights)+1),squeeze=False,
                          gridspec_kw={'height_ratios':heights,'width_ratios':[1,1]})
    names={'pretraining':'Complete JEPA training update','encoder_forward':'Encoder forward',
           'feature_extraction':'Feature extraction','probe_fitting':'Frozen linear-probe fitting'}
    for index,phase in enumerate(active):
        selected=grouped[phase];left,right=axes[index];labels=[_performance_label(r) for r in selected]
        for y,row in enumerate(selected):
            kind=_kind(row);color=COLORS.get(kind,'#666666');initial='untrained' in str(row.get('model_id',''))
            value,lo,hi=_timing_value(row,phase)
            if value is not None and np.isfinite(value):
                error=np.array([[lo],[hi]]) if lo is not None and hi is not None else None
                bar=left.barh(y,value,color='white' if initial else color,edgecolor=color,hatch='///' if initial else None,
                              xerr=error,error_kw={'capsize':3,'linewidth':1,'ecolor':'#333333'})
                left.bar_label(bar,labels=[f'{value:.3g}'],padding=5,fontsize=9)
            else:left.text(.02,y,'NOT RUN',transform=left.get_yaxis_transform(),va='center',color='#666666')
            memory=row.get('peak_allocated_bytes',row.get('peak_cuda_allocated_bytes'))
            if memory is not None and row.get('status') not in ('NOT_RUN','FAIL'):
                bar=right.barh(y,float(memory)/2**20,color='white' if initial else color,edgecolor=color,hatch='///' if initial else None)
                right.bar_label(bar,fmt='%.1f',padding=5,fontsize=9)
            else:right.text(.02,y,'NOT RUN / CPU peak unmeasured',transform=right.get_yaxis_transform(),va='center',color='#666666',fontsize=9)
        unit='Median update time (seconds; whiskers = IQR)' if phase=='pretraining' else 'Median forward time (ms; whiskers = MAD)' if phase=='encoder_forward' else 'Total stage wall time (seconds)'
        devices=', '.join(sorted({str(r.get('device','unrecorded')) for r in selected}))
        left.set_title(f'{names[phase]} · {devices}',loc='left',weight='bold',fontsize=11)
        right.set_title('Peak CUDA allocated memory',loc='left',fontsize=11)
        for ax in (left,right):
            ax.set_yticks(range(len(labels)),labels);ax.set_ylim(max(.5,len(labels)-.5),-.5);ax.grid(axis='x',alpha=.2);ax.set_xlim(left=0)
            if selected:ax.set_xlim(0,max(ax.get_xlim()[1],1e-9)*1.22)
            else:ax.text(.5,.5,'NOT RUN',transform=ax.transAxes,ha='center')
        left.set_xlabel(unit);right.set_xlabel('MiB; includes live model/input tensors, excludes reserved cache')
    label='Smoke: integration-only' if run_mode=='smoke' else 'Pilot: one paired seed'
    fig.suptitle(f'Measured execution stages — {label}',fontsize=14)
    fig.text(.5,.006,'Compare models within the same stage and workload. Stage totals include different operations; see performance tables for timing scope.\nIQR/MAD show timing dispersion, not confidence intervals. Missing measurements are never zero.',ha='center',fontsize=10)
    fig.tight_layout(rect=(0,.047,1,.975),h_pad=2,w_pad=3);return fig

def comparison_figure(comparison):
    """Conditional paired-bootstrap intervals, not across-training-seed error bars."""
    fig,axes=plt.subplots(1,2,figsize=(11,4.5))
    names=[('delta_vit','ViT trained − untrained'),('delta_fmm','FMM trained − untrained'),('delta_architecture','Trained FMM − trained ViT')]
    for ax,metric in zip(axes,('correct_top1','correct_top5')):
        for i,(name,label) in enumerate(names):
            value=comparison.get('deltas',{}).get(name,{}).get(metric)
            if value is None:ax.text(0,i,'NOT RUN',va='center');continue
            point=value['delta_percentage_points'];lo,hi=value['ci95_percentage_points']
            ax.hlines(i,lo,hi,color='#245b80',linewidth=1.5);ax.plot([lo,hi],[i,i],'|',color='#245b80',markersize=8)
            ax.plot(point,i,'o',color='#245b80')
            ax.annotate(f'{point:+.2f}',(point,i),xytext=(0,10),textcoords='offset points',ha='center',fontsize=9)
        ax.axvline(0,color='#444444',linestyle='--',linewidth=1);ax.set_yticks(range(3),[label for _,label in names]);ax.invert_yaxis()
        ax.set_xlabel(('Top-1' if metric=='correct_top1' else 'Top-5')+' accuracy difference (percentage points)');ax.grid(axis='x',alpha=.2)
    fig.suptitle('Paired evaluation-image bootstrap: 95% intervals conditional on these fixed models')
    caveat='Smoke learning remains inconclusive.' if comparison.get('run_mode','smoke')=='smoke' else 'One paired training seed.'
    fig.text(.5,.02,'Development-selected probes; intervals exclude training-seed variability. '+caveat,ha='center',fontsize=10)
    fig.tight_layout(rect=(0,.07,1,.95));return fig


## Bounded renderer check

The following cell uses visibly labeled synthetic shape fixtures solely to verify the plotting interfaces and missing-data handling. It does not perform evaluation or register any measurement. Notebook 01 supplies actual saved run rows.


In [ ]:
# Shape/render validation only. These are explicitly synthetic fixtures, never run measurements.
setup_style()
_fixture_training=[]
for _kind_name in ('vit','fmm'):
    for _step_number in (1,2,3):
        _fixture_training.append({'model_id':_kind_name,'successful_updates':_step_number,'loss':.8/_step_number,
          'gradient_norm_preclip':1./_step_number,'near_norm':.2,'far_norm':.03,
          'near_gradient_norm_postclip':.05,'far_gradient_norm_postclip':.001})
_fixture_diagnostics=[{'model_id':kind,'step':step,'results':{'feature_std_mean':.1+step*.001,
  'between_image_cosine_mean':.7,'effective_rank':30,'matching_patch_across_images_std_mean':.2,
  'diagnostic_examples':200,'effective_rank_ceiling':192}} for kind in ('vit','fmm') for step in (0,10,20)]
_fixture_controls={kind+'_'+state:{'summary':{'metrics':{'top1':.01,'top5':.04,'examples':1000}}}
                   for kind in ('vit','fmm') for state in ('untrained','trained')}
_fixture_performance=[]
for _kind_name in ('vit','fmm'):
    _fixture_performance.extend([
        {'model_id':_kind_name,'phase':'pretraining','device':'cuda:0','complete_jepa_step_seconds':{'median':.1,'p25':.09,'p75':.12},'peak_cuda_allocated_bytes':100*2**20},
        {'model_id':_kind_name,'phase':'encoder_forward','device':'cuda:0','median_seconds':.004,'mad_seconds':.001,'peak_allocated_bytes':70*2**20}])
    for _state in ('untrained','trained'):
        for _split in ('train','development'):
            _fixture_performance.append({'model_id':_kind_name+'_'+_state,'split':_split,'phase':'feature_extraction','device':'cuda:0','wall_seconds':2.,'peak_allocated_bytes':80*2**20})
        _fixture_performance.append({'model_id':_kind_name+'_'+_state,'phase':'probe_fitting','device':'cpu','wall_seconds':1.,'peak_allocated_bytes':None})
_fixture_figures=[training_figure(_fixture_training),diagnostics_figure(_fixture_diagnostics),probe_figure(_fixture_controls),performance_figure(_fixture_performance)]
for _figure in _fixture_figures:
    _figure.suptitle('Synthetic fixture: rendering verification only; no experiment evidence',fontsize=13)
    _figure.canvas.draw()
assert len(_fixture_figures[0].axes[0].lines)==2
assert len(_fixture_figures[1].axes[0].lines)==2
assert len(_fixture_figures[2].axes[0].patches)==4
assert len(_fixture_figures[3].axes)==8
_fixture_directory=PROJECT_ROOT/'artifacts'/'evaluation_unit_fixtures'
_fixture_directory.mkdir(parents=True,exist_ok=True)
_fixture_figures[-1].savefig(_fixture_directory/'performance_render_fixture.png',dpi=95,bbox_inches='tight')
for _figure in _fixture_figures:plt.close(_figure)
print('PASS: training, nested diagnostics, four full probe summaries, encoder timing schema, and explicit missing CPU memory render correctly. Synthetic shape fixtures only.')
